# Milestone 3, Person A: contract text → clause detections (pipeline stages 1–4)

This notebook is the first half of the end-to-end pipeline. It answers **"what clauses are in this contract, and where?"**

| Stage | What happens here |
|---|---|
| 1. Clean | Same text cleaning as the October notebook |
| 2. Chunk | Same 450-token windows with 50-token overlap the model was trained on |
| 3. Classify | Fine-tuned `bert-base-uncased` labels each chunk with any of CUAD's 41 clause categories |
| 4. Evidence | Each detection keeps the **original contract text** and its character position |

**Output (the hand-off to Person B):** `detections.json`, with one entry per contract listing every detected clause, its evidence text, and the model's confidence.

### How to run
1. Open in **Google Colab** and choose *Runtime → Change runtime type → GPU (T4 or better)*.
2. To check that everything works first, set `SMOKE_TEST = True` in the Settings cell and run all cells. This takes a few minutes and the numbers it produces are meaningless.
3. Set `SMOKE_TEST = False` and run all cells. **The first full run trains the model (roughly 30–60 minutes on a T4)** and saves it to Google Drive. Later runs load the saved model and skip training.

> **Why 450/50 and not 512/64?** The advisor suggested 512 tokens with 64 overlap, but the October model was trained on 450-token chunks with 50-token overlap. The model must see the same kind of input it was trained on, so this notebook keeps 450/50. BERT adds 2 special tokens, so a chunk still fits in BERT's 512 limit.

## 0. Settings

In [ ]:
# ---------------- Settings you may change ----------------
SMOKE_TEST = False      # True = tiny, fast run to check the notebook works (results are meaningless)
FORCE_RETRAIN = False   # True = train a new model even if a saved one exists

MODEL_NAME = "bert-base-uncased"   # same base model as October
CHUNK_SIZE = 450                   # tokens per chunk (must match training)
OVERLAP = 50                       # tokens shared between neighbouring chunks
MAX_LENGTH = 512                   # BERT's input limit (chunk + special tokens)

EPOCHS = 3
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
GRAD_ACCUM = 2                     # effective batch size = BATCH_SIZE * GRAD_ACCUM
SEED = 42

# Class-imbalance weights (Kenneth's October method): weight = negatives / positives per category.
# Very rare categories can get weights in the hundreds or thousands. Set a number (e.g. 50) to cap them
# if training becomes unstable or floods you with false positives. None = no cap (October behaviour).
MAX_POS_WEIGHT = None

# Thresholds: how confident the model must be before we call a clause "detected".
# They are tuned per category on the VALIDATION contracts. THRESHOLD_BETA = 1 balances precision and
# recall (F1). THRESHOLD_BETA = 2 favours recall (catch more clauses, accept more false alarms), which
# suits the project goal of not missing risky contracts.
THRESHOLD_BETA = 1.0

RUN_ON_SPLIT = "test"   # which contracts to produce detections for: "test" or "val"
# ----------------------------------------------------------
STEP_SIZE = CHUNK_SIZE - OVERLAP

## Setup: where the data comes from and where results are saved
In Colab, this cell mounts Google Drive, where the trained model and outputs are kept so they survive the session ending. It also clones the team repo to get the data.

If the clone fails (for example because the repo is private), upload the repo folder to Drive and set `REPO_DIR` to its path.

In [ ]:
import importlib.util, subprocess, sys
missing = [p for p in ["transformers", "accelerate"] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import json, os, re, random, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from transformers.utils import logging as hf_logging
from sklearn.preprocessing import MultiLabelBinarizer

hf_logging.set_verbosity_error()   # hides the harmless "sequence longer than 512" warning when chunking

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1L-contract-review-challenge.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/Accenture-1L-contract-review-challenge")
    if not REPO_DIR.exists():
        result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
        if result.returncode != 0:
            raise RuntimeError("Could not clone the repo. Upload it to Drive and set REPO_DIR to that folder.")
    OUTPUT_DIR = Path("/content/drive/MyDrive/contract_review_pipeline")
    CHECKPOINT_DIR = Path("/content/training_checkpoints")      # temporary, not on Drive
else:
    REPO_DIR = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
    OUTPUT_DIR = REPO_DIR / "outputs" / "pipeline"
    CHECKPOINT_DIR = OUTPUT_DIR / "training_checkpoints"

if SMOKE_TEST:
    OUTPUT_DIR = OUTPUT_DIR / "smoke_test"      # never mix smoke-test files with real results
MODEL_DIR = OUTPUT_DIR / "model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("Repo:      ", REPO_DIR)
print("Outputs:   ", OUTPUT_DIR)
print("Device:    ", DEVICE)
if DEVICE.type == "cpu" and not SMOKE_TEST:
    print("WARNING: no GPU found. Training on CPU will take many hours. In Colab: Runtime -> Change runtime type -> GPU.")

## Stage 1: Load and clean the contracts
This uses the same train/validation/test contract files the team made in September (`notebooks/*_contracts.csv`) and the **same cleaning function**: it tidies line endings, repeated spaces and big blank gaps, and changes no legal wording.

- **Train** contracts teach the model.
- **Validation** contracts pick the confidence thresholds.
- **Test** contracts are the ones we produce detections for. The model never saw them during training.

In [ ]:
def clean_contract_text(text):
    """Identical to the October notebook: normalise line endings, collapse spaces, trim huge gaps."""
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

SPLITS = ["train", "val", "test"]
contracts = {s: pd.read_csv(REPO_DIR / "notebooks" / f"{s}_contracts.csv") for s in SPLITS}
annotations = {s: pd.read_csv(REPO_DIR / "notebooks" / f"{s}_annotations.csv") for s in SPLITS}
for s in SPLITS:
    contracts[s]["context"] = contracts[s]["context"].fillna("").apply(clean_contract_text)

# The original (uncleaned) CUAD text is needed to locate each expert-highlighted clause.
with open(REPO_DIR / "data" / "CUADv1.json", encoding="utf-8") as f:
    original_contexts = {c["title"]: c["paragraphs"][0]["context"] for c in json.load(f)["data"]}

# No contract may appear in two splits, or evaluation numbers would be inflated.
titles = {s: set(contracts[s]["contract_title"]) for s in SPLITS}
assert not (titles["train"] & titles["val"]) and not (titles["train"] & titles["test"]) and not (titles["val"] & titles["test"]), \
    "A contract appears in more than one split."

if SMOKE_TEST:
    contracts = {s: df.head(4 if s != "test" else 3).reset_index(drop=True) for s, df in contracts.items()}
    annotations = {s: a[a["contract_title"].isin(contracts[s]["contract_title"])] for s, a in annotations.items()}

for s in SPLITS:
    print(f"{s:5s}: {len(contracts[s]):3d} contracts, {len(annotations[s]):5d} expert-labelled clauses")

## Stage 2: Cut each contract into chunks
BERT can only read about 512 tokens at once, so each contract is cut into overlapping 450-token windows. This is the same logic as the October `chunk_contract` function.

The new part: for every chunk we also record **where it sits in the original contract** (`char_start`, `char_end`). The model reads a lower-cased version of the text, but the evidence handed to Person B is the **exact original contract wording**. This matters because Katie's risk scorer requires evidence that matches the contract text exactly.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def chunk_contract(text):
    """Split one contract into overlapping token windows.

    Returns a list of dicts. `model_text` is exactly what October training fed to BERT;
    `evidence_text` is the matching slice of the original contract (for humans and the scorer)."""
    enc = tokenizer(text, add_special_tokens=False, truncation=False, return_offsets_mapping=True)
    ids, offsets = enc["input_ids"], enc["offset_mapping"]
    chunks = []
    for start in range(0, len(ids), STEP_SIZE):
        window = ids[start:start + CHUNK_SIZE]
        # Same decode -> re-encode -> decode steps as the October notebook, so model input matches training.
        model_text = tokenizer.decode(window, skip_special_tokens=True)
        final = tokenizer.encode(model_text, add_special_tokens=False, truncation=True, max_length=CHUNK_SIZE)
        model_text = tokenizer.decode(final, skip_special_tokens=True)

        end = min(start + CHUNK_SIZE, len(ids))
        char_start, char_end = offsets[start][0], offsets[end - 1][1]
        chunks.append({"chunk_id": len(chunks), "char_start": char_start, "char_end": char_end,
                       "model_text": model_text, "evidence_text": text[char_start:char_end]})
        if start + CHUNK_SIZE >= len(ids):
            break
    return chunks

def chunk_split(contracts_df):
    rows = []
    for _, row in contracts_df.iterrows():
        for chunk in chunk_contract(row["context"]):
            rows.append({"contract_title": row["contract_title"], **chunk})
    return pd.DataFrame(rows)

chunks = {s: chunk_split(contracts[s]) for s in SPLITS}
for s in SPLITS:
    print(f"{s:5s}: {len(chunks[s]):6d} chunks from {len(contracts[s])} contracts")

# Sanity check: evidence must be an exact slice of the cleaned contract.
sample = chunks["test"].iloc[0]
full = contracts["test"].set_index("contract_title").loc[sample["contract_title"], "context"]
assert full[sample["char_start"]:sample["char_end"]] == sample["evidence_text"]
print("\nFirst test chunk (original wording), first 300 characters:\n", sample["evidence_text"][:300])

## Stage 3a: Give each chunk its correct answers (for training and checking)
To train the model and measure how good it is, each chunk needs the right answer: which clause categories the CUAD lawyers highlighted inside it. This is the October `categorise_chunks` logic: find each highlighted clause in the cleaned text, then attach its category to every chunk it overlaps.

This step is **only** for training, threshold tuning and the accuracy check at the end. Brand-new contracts would have no expert labels, and the pipeline would skip this step.

In [ ]:
def label_chunks(chunks_df, annotations_df):
    """Attach the list of expert-labelled categories overlapping each chunk (October logic)."""
    result = chunks_df.copy().reset_index(drop=True)
    labels = [[] for _ in range(len(result))]
    unmatched = 0
    for title, idx in result.groupby("contract_title").groups.items():
        original_text = original_contexts[title]
        text = cleaned_texts[title]          # the exact text the chunk positions refer to
        clauses = []
        anns = annotations_df[annotations_df["contract_title"] == title].drop_duplicates(
            subset=["category", "answer_start", "clause_text"])
        for _, a in anns.iterrows():
            clause = clean_contract_text(str(a["clause_text"]))
            expected = len(clean_contract_text(original_text[:int(a["answer_start"])]))
            start = text.find(clause, max(0, expected - 3), expected + len(clause) + 4)
            if not clause or start == -1:
                unmatched += 1
                continue
            clauses.append((a["category"], start, start + len(clause)))
        for i in idx:
            cs, ce = result.at[i, "char_start"], result.at[i, "char_end"]
            labels[i] = sorted({cat for cat, s, e in clauses if s < ce and e > cs})
    result["categories"] = labels
    return result, unmatched

cleaned_texts = {t: c for s in SPLITS for t, c in zip(contracts[s]["contract_title"], contracts[s]["context"])}
for s in SPLITS:
    chunks[s], n_unmatched = label_chunks(chunks[s], annotations[s])
    with_labels = (chunks[s]["categories"].str.len() > 0).sum()
    print(f"{s:5s}: {with_labels:6d} of {len(chunks[s]):6d} chunks contain at least one labelled clause "
          f"({n_unmatched} highlighted clauses could not be located)")

# The category list. Same as October: every category that appears in the training chunks.
import warnings
warnings.filterwarnings("ignore", message="unknown class")   # categories never seen in training are dropped (as in October)
mlb = MultiLabelBinarizer()
y = {"train": mlb.fit_transform(chunks["train"]["categories"])}
LABELS = list(mlb.classes_)
for s in ["val", "test"]:
    y[s] = mlb.transform(chunks[s]["categories"])
print(f"\n{len(LABELS)} categories:", ", ".join(LABELS))

## Stage 3b: Get a trained model (load it if saved, otherwise train it)
If a model is already saved in `OUTPUT_DIR/model`, it is loaded and training is skipped.

Otherwise the model is trained using Kenneth's **weighted loss** from October. Rare clause types count for more during training, so the model doesn't learn to just say "nothing here" (which is why the October 1-epoch run scored zero). After every epoch the model is checked against the validation contracts, and the best version is kept.

In [ ]:
class ChunkDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, padding="max_length", max_length=MAX_LENGTH)
        self.labels = labels
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[i], dtype=torch.float32)
        return item

class WeightedMultilabelTrainer(Trainer):
    """Kenneth's October trainer: BCE loss with a positive-class weight per category."""
    def __init__(self, pos_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        loss = self.loss_fn(outputs.logits, labels.to(outputs.logits.dtype))
        return (loss, outputs) if return_outputs else loss

def micro_f1_metric(eval_pred):
    logits, labels = eval_pred
    preds = (1 / (1 + np.exp(-logits))) >= 0.5
    tp = (preds & (labels == 1)).sum(); fp = (preds & (labels == 0)).sum(); fn = (~preds & (labels == 1)).sum()
    return {"micro_f1": float(2 * tp / (2 * tp + fp + fn)) if tp + fp + fn else 0.0}

saved_labels_file = MODEL_DIR / "labels.json"
if (MODEL_DIR / "config.json").exists() and saved_labels_file.exists() and not FORCE_RETRAIN:
    print("Loading saved model from", MODEL_DIR)
    saved_labels = json.loads(saved_labels_file.read_text(encoding="utf-8"))
    assert saved_labels == LABELS, "Saved model was trained on a different category list. Set FORCE_RETRAIN = True."
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
else:
    print("No saved model found (or FORCE_RETRAIN). Training a new one...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), problem_type="multi_label_classification",
        id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)})

    pos = y["train"].sum(axis=0)
    pos_weights = (len(y["train"]) - pos) / np.maximum(pos, 1)
    if MAX_POS_WEIGHT is not None:
        pos_weights = np.minimum(pos_weights, MAX_POS_WEIGHT)
    print("Rarest categories get the biggest weights:")
    print(pd.Series(pos_weights, index=LABELS).sort_values(ascending=False).head(5).round(1).to_string())

    train_ds = ChunkDataset(chunks["train"]["model_text"], y["train"])
    val_ds = ChunkDataset(chunks["val"]["model_text"], y["val"])
    args = TrainingArguments(
        output_dir=str(CHECKPOINT_DIR), eval_strategy="epoch", save_strategy="epoch",
        learning_rate=LEARNING_RATE, per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS, weight_decay=0.01, logging_steps=50,
        load_best_model_at_end=True, metric_for_best_model="micro_f1", greater_is_better=True,
        save_total_limit=1, fp16=torch.cuda.is_available(), report_to="none", seed=SEED,
        max_steps=4 if SMOKE_TEST else -1)
    trainer = WeightedMultilabelTrainer(
        pos_weights=torch.tensor(pos_weights, dtype=torch.float32, device=DEVICE),
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=micro_f1_metric)
    trainer.train()
    model = trainer.model.to(DEVICE)

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_DIR)
    tokenizer.save_pretrained(MODEL_DIR)
    saved_labels_file.write_text(json.dumps(LABELS, indent=2), encoding="utf-8")
    (MODEL_DIR / "training_settings.json").write_text(json.dumps({
        "base_model": MODEL_NAME, "chunk_size": CHUNK_SIZE, "overlap": OVERLAP, "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
        "max_pos_weight": MAX_POS_WEIGHT, "seed": SEED, "smoke_test": SMOKE_TEST,
        "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "best_val_micro_f1_at_0.5": trainer.state.best_metric}, indent=2), encoding="utf-8")
    print("Saved model to", MODEL_DIR)
model.eval();

## Stage 3c: Run the model on every chunk
For every chunk and every category, the model outputs a **probability** between 0 and 1: its confidence that the chunk contains that clause type.

In [ ]:
@torch.no_grad()
def predict_probabilities(texts, batch_size=32):
    ds = ChunkDataset(texts)
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size)
    out = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits = model(**batch).logits
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(LABELS)))

probs = {}
for s in ["val", RUN_ON_SPLIT]:
    if s not in probs:
        probs[s] = predict_probabilities(chunks[s]["model_text"])
        print(f"{s}: probabilities for {probs[s].shape[0]} chunks x {probs[s].shape[1]} categories")

## Stage 3d: Choose a confidence threshold for each category
A probability becomes a yes/no detection when it passes a **threshold**. One fixed 0.5 cut-off works badly when some clause types are common and others appear in only a handful of contracts, so each category gets its own threshold, chosen on the **validation** contracts (never the test contracts).

- A **lower** threshold catches more real clauses but raises more false alarms.
- Categories with no examples in the validation contracts keep 0.5.

In [ ]:
def f_beta(tp, fp, fn, beta):
    b2 = beta ** 2
    denom = (1 + b2) * tp + b2 * fn + fp
    return (1 + b2) * tp / denom if denom else 0.0

CANDIDATES = np.round(np.arange(0.05, 0.96, 0.05), 2)
thresholds, threshold_notes = {}, {}
for j, label in enumerate(LABELS):
    truth = y["val"][:, j] == 1
    if truth.sum() == 0:
        thresholds[label], threshold_notes[label] = 0.5, "no validation examples; default 0.5"
        continue
    scores = [f_beta(((probs["val"][:, j] >= t) & truth).sum(), ((probs["val"][:, j] >= t) & ~truth).sum(),
                     ((probs["val"][:, j] < t) & truth).sum(), THRESHOLD_BETA) for t in CANDIDATES]
    best = int(np.argmax(scores))
    if scores[best] == 0:
        thresholds[label], threshold_notes[label] = 0.5, "model never correct on validation; default 0.5"
    else:
        thresholds[label], threshold_notes[label] = float(CANDIDATES[best]), f"tuned on validation (F{THRESHOLD_BETA:g}={scores[best]:.2f})"

(OUTPUT_DIR / "thresholds.json").write_text(json.dumps(
    {"beta": THRESHOLD_BETA, "chosen_on": "val", "thresholds": thresholds, "notes": threshold_notes}, indent=2), encoding="utf-8")
print(pd.DataFrame({"threshold": thresholds, "how chosen": threshold_notes}).to_string())

## Stage 4: Build the hand-off file for Person B
For each contract in `RUN_ON_SPLIT`, a detection is recorded for every chunk where a category's probability passes its threshold. Each detection carries:

- `category`: the clause type the model found
- `text`, `char_start`, `char_end`: the **original wording** of the chunk and where it sits in the cleaned contract
- `probability` and `threshold`: how confident the model was, and the cut-off it passed

`chunks_scanned` and `categories_checked` tell Person B the **whole contract** was checked for those categories. Without that, nobody can safely say a clause is *missing*.

> A detection says "this clause type is somewhere in this chunk". It does **not** say whether the clause is risky. That judgement is Person B's stage 5.

In [ ]:
def build_detections(split):
    df, p = chunks[split], probs[split]
    out = []
    for title, idx in df.groupby("contract_title", sort=True).groups.items():
        idx = list(idx)
        detections = []
        for i in idx:
            for j, label in enumerate(LABELS):
                if p[i, j] >= thresholds[label]:
                    detections.append({
                        "category": label,
                        "chunk_id": int(df.at[i, "chunk_id"]),
                        "char_start": int(df.at[i, "char_start"]),
                        "char_end": int(df.at[i, "char_end"]),
                        "text": df.at[i, "evidence_text"],
                        "probability": round(float(p[i, j]), 4),
                        "threshold": thresholds[label],
                    })
        detections.sort(key=lambda d: (d["category"], -d["probability"]))
        out.append({"contract_id": title, "split": split, "chunks_scanned": len(idx),
                    "categories_detected": sorted({d["category"] for d in detections}),
                    "detections": detections})
    # Contracts that produced zero chunks (empty text) are still listed, so nothing silently disappears.
    for title in sorted(set(contracts[split]["contract_title"]) - set(df["contract_title"])):
        out.append({"contract_id": title, "split": split, "chunks_scanned": 0,
                    "categories_detected": [], "detections": []})
    return out

handoff = {
    "schema_version": 1,
    "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "smoke_test": SMOKE_TEST,
    "model": {"base_model": MODEL_NAME, "model_dir": str(MODEL_DIR), "chunk_size": CHUNK_SIZE,
              "overlap": OVERLAP, "threshold_beta": THRESHOLD_BETA, "thresholds_chosen_on": "val"},
    "text_note": "text/char_start/char_end refer to the contract after clean_contract_text().",
    "categories_checked": LABELS,
    "contracts": build_detections(RUN_ON_SPLIT),
}
out_file = OUTPUT_DIR / f"detections_{RUN_ON_SPLIT}.json"
out_file.write_text(json.dumps(handoff, indent=2, ensure_ascii=False), encoding="utf-8")

# Also save every chunk's probabilities, useful for Milestone 2 evaluation and for re-thresholding later.
prob_df = chunks[RUN_ON_SPLIT][["contract_title", "chunk_id", "char_start", "char_end"]].copy()
prob_df[LABELS] = probs[RUN_ON_SPLIT]
prob_df.to_csv(OUTPUT_DIR / f"chunk_probabilities_{RUN_ON_SPLIT}.csv", index=False, encoding="utf-8")

n_det = sum(len(c["detections"]) for c in handoff["contracts"])
print(f"Saved {out_file}")
print(f"{len(handoff['contracts'])} contracts, {n_det} detections")

### Look at one contract
This is what Person B receives for a single contract. Only the advisor's priority categories are shown here, to keep it short.

In [ ]:
PRIORITY = ["Cap On Liability", "Uncapped Liability", "Termination For Convenience", "Governing Law",
            "Renewal Term", "Notice Period To Terminate Renewal", "Ip Ownership Assignment",
            "Change Of Control", "Non-Compete", "Audit Rights", "Most Favored Nation"]
example = max(handoff["contracts"], key=lambda c: len(c["categories_detected"]))
print("Contract:", example["contract_id"])
print("Chunks scanned:", example["chunks_scanned"])
print("Categories detected:", ", ".join(example["categories_detected"]) or "(none)")
print()
for d in example["detections"]:
    if d["category"] in PRIORITY:
        print(f"- {d['category']}  (chunk {d['chunk_id']}, p={d['probability']:.2f}, threshold {d['threshold']})")
        print("   ", d["text"][:200].replace("\n", " "), "...")

## Check: how much can Person B trust this?
The test contracts have expert labels, so we can measure the detections against them. There are two views:

1. **Chunk level:** per category, precision (when it says "found", how often it's right) and recall (of the real clauses, how many it found).
2. **Contract level** (the one that matters most for risk): for each category, out of the contracts that really contain that clause, how often did the model find it **anywhere** in the contract? When it doesn't, Person B could wrongly flag the clause as **missing**. The `false_missing_rate` column shows how often that would happen.

Use these numbers to decide which categories Person B can rely on and which should be marked `unknown` for a human to check.

In [ ]:
split = RUN_ON_SPLIT
pred = np.column_stack([probs[split][:, j] >= thresholds[l] for j, l in enumerate(LABELS)])
truth = y[split] == 1

rows = []
contract_ids = chunks[split]["contract_title"].values
for j, label in enumerate(LABELS):
    tp = int((pred[:, j] & truth[:, j]).sum()); fp = int((pred[:, j] & ~truth[:, j]).sum())
    fn = int((~pred[:, j] & truth[:, j]).sum())
    has = pd.Series(truth[:, j]).groupby(contract_ids).any()
    found = pd.Series(pred[:, j]).groupby(contract_ids).any()
    n_has = int(has.sum())
    rows.append({
        "category": label, "priority": label in PRIORITY,
        "chunk_precision": round(tp / (tp + fp), 3) if tp + fp else None,
        "chunk_recall": round(tp / (tp + fn), 3) if tp + fn else None,
        "contracts_with_clause": n_has,
        "contract_recall": round(float((has & found).sum()) / n_has, 3) if n_has else None,
        "false_missing_rate": round(float((has & ~found).sum()) / n_has, 3) if n_has else None,
        "contracts_flagged_without_clause": int((~has & found).sum()),
    })
report = pd.DataFrame(rows).sort_values(["priority", "contracts_with_clause"], ascending=[False, False])
report.to_csv(OUTPUT_DIR / f"detection_quality_{split}.csv", index=False, encoding="utf-8")

micro_tp = int((pred & truth).sum()); micro_fp = int((pred & ~truth).sum()); micro_fn = int((~pred & truth).sum())
print(f"Overall chunk-level ({split}): precision {micro_tp / max(micro_tp + micro_fp, 1):.3f}, "
      f"recall {micro_tp / max(micro_tp + micro_fn, 1):.3f}, "
      f"F1 {2 * micro_tp / max(2 * micro_tp + micro_fp + micro_fn, 1):.3f}")
if SMOKE_TEST:
    print("SMOKE TEST: the model barely trained, so these numbers are meaningless.")
report

## Done: what to give Person B
Files in `OUTPUT_DIR` (on Google Drive when run in Colab):

| File | What it is |
|---|---|
| `detections_test.json` | **The hand-off.** Detected clauses with original evidence text, for each contract |
| `detection_quality_test.csv` | How reliable each category is. Share it with Person B so they know which detections to trust |
| `thresholds.json` | The confidence cut-off used for each category |
| `chunk_probabilities_test.csv` | Every chunk's raw probabilities (for evaluation or re-thresholding) |
| `model/` | The trained model. Later runs reuse it instead of retraining |

**Before sharing:** check that `smoke_test` is `false` at the top of `detections_test.json`.